In [ ]:
import pandas as pd

df = pd.read_csv("users_data.csv")

print("Shape:", df.shape)
print(df.head())

df["gender"] = df["gender"].str.lower().str.strip()
df["activity_level"] = df["activity_level"].str.lower().str.strip()
df["goal"] = df["goal"].str.lower().str.strip()

allowed_gender = {"male", "female"}
allowed_activity = {"low", "moderate", "high"}
allowed_goal = {"fat_loss", "muscle_gain", "recomposition"}

assert df["gender"].isin(allowed_gender).all()
assert df["activity_level"].isin(allowed_activity).all()
assert df["goal"].isin(allowed_goal).all()

def calc_bmr(row):
    w = row["weight_kg"]
    h = row["height_cm"]
    a = row["age"]
    if row["gender"] == "male":
        return 10*w + 6.25*h - 5*a + 5
    else:
        return 10*w + 6.25*h - 5*a - 161

activity_map = {
    "low": 1.2,
    "moderate": 1.55,
    "high": 1.725
}

def calc_tdee(row):
    return row["BMR"] * activity_map[row["activity_level"]]

def calc_target_calories(row):
    tdee = row["TDEE"]
    if row["goal"] == "fat_loss":
        return tdee - 500
    elif row["goal"] == "muscle_gain":
        return tdee + 300
    else:  # recomposition
        return tdee - 200

def calc_protein(row):
    w = row["weight_kg"]
    if row["goal"] == "fat_loss":
        return round(2.0 * w)   
    elif row["goal"] == "muscle_gain":
        return round(2.2 * w)   
    else:
        return round(2.0 * w)

df["BMR"] = df.apply(calc_bmr, axis=1)
df["TDEE"] = df.apply(calc_tdee, axis=1)
df["target_calories"] = df.apply(calc_target_calories, axis=1)
df["protein_g"] = df.apply(calc_protein, axis=1)

print(df[["user_id","BMR","TDEE","target_calories","protein_g"]].head())

df.to_csv("users_data_enriched.csv", index=False)
print("Saved: users_data_enriched.csv")


Shape: (100, 7)
   user_id  age  gender  weight_kg  height_cm activity_level           goal
0        1   24  female         79        151            low       fat_loss
1        2   37  female         95        171            low       fat_loss
2        3   32  female         54        152            low       fat_loss
3        4   28    male         68        188           high  recomposition
4        5   25  female         65        178           high       fat_loss
   user_id      BMR       TDEE  target_calories  protein_g
0        1  1452.75  1743.3000        1243.3000        158
1        2  1672.75  2007.3000        1507.3000        190
2        3  1169.00  1402.8000         902.8000        108
3        4  1720.00  2967.0000        2767.0000        136
4        5  1476.50  2546.9625        2046.9625        130
✅ Saved: users_data_enriched.csv
